In [1]:
#Quinlin Neuhaus
#MATH 5001 Deep Learning
#Code is taken from provided notebook and my own adaptations

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
def get_best_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_best_device()
print("Using device:", device)


Using device: cuda


In [4]:
trainset = datasets.MNIST(root='./data', train=True, download=True)

temp1 = trainset.data.float().mean().numpy()
temp2 = trainset.data.float().std().numpy()

print('Min Pixel Value:', trainset.data.min().numpy())
print('Max Pixel Value:', trainset.data.max().numpy())
print('Mean Pixel Value:', temp1)
print('Pixel Values Std:', temp2)
print('Scaled Mean Pixel Value:', temp1 / 255)
print('Scaled Pixel Values Std:', temp2 / 255)


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.17MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 136kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.0MB/s]


Min Pixel Value: 0
Max Pixel Value: 255
Mean Pixel Value: 33.31842
Pixel Values Std: 78.56749
Scaled Mean Pixel Value: 0.13066047
Scaled Pixel Values Std: 0.3081078


In [5]:
batch_size = 64

transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST(root="./data",train=True,download=True,transform=transform)
test_dataset = datasets.MNIST(root="./data",train=False,download=True,transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

shuffled_train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
targets_tensor = torch.tensor(shuffled_train_dataset.targets)
shuffle_index = torch.randperm(len(targets_tensor))
shuffled_targets = targets_tensor[shuffle_index]
shuffled_train_dataset.targets = shuffled_targets

shuffled_train_loader = DataLoader(shuffled_train_dataset, batch_size=256, shuffle=True)


/tmp/ipython-input-2218495471.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  targets_tensor = torch.tensor(shuffled_train_dataset.targets)


In [6]:
class MNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1)

        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:

model = MNISTCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [7]:
def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [8]:
def test_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return correct / total

In [ ]:
%%time

epochs = 10

for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer)
    acc = test_accuracy(model, test_loader)

    print(f"Epoch {epoch:2d} | Loss: {loss:.4f} | Test Acc: {acc:.4f}")

torch.save(model.state_dict(), "mnist_minivgg.pt")

Epoch  1 | Loss: 0.1312 | Test Acc: 0.9845
Epoch  2 | Loss: 0.0408 | Test Acc: 0.9914
Epoch  3 | Loss: 0.0266 | Test Acc: 0.9885
Epoch  4 | Loss: 0.0211 | Test Acc: 0.9904
Epoch  5 | Loss: 0.0176 | Test Acc: 0.9893
Epoch  6 | Loss: 0.0134 | Test Acc: 0.9916
Epoch  7 | Loss: 0.0116 | Test Acc: 0.9913
Epoch  8 | Loss: 0.0106 | Test Acc: 0.9934
Epoch  9 | Loss: 0.0086 | Test Acc: 0.9927
Epoch 10 | Loss: 0.0078 | Test Acc: 0.9926
CPU times: user 3min 37s, sys: 1.09 s, total: 3min 38s
Wall time: 3min 43s


In [ ]:
torch.save(model.state_dict(), "mnist_minivgg.pt")

In [ ]:
%%time
#Question 3 HyperParameters testing:
batch_sizes = range(32,96+1, 16)
dnn_sizes = range(32,128+1, 32)
acc_list = []

transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.MNIST(root="./data",train=True,download=True,transform=transform)
test_dataset = datasets.MNIST(root="./data",train=False,download=True,transform=transform)

for i in batch_sizes:
  batch_size = i

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)
  for j in dnn_sizes:
    model = MNISTCNN().to(device)
    model.fc1 = nn.Linear(64 * 7 * 7, j).to(device) # Move to device
    model.fc2 = nn.Linear(j, 10).to(device) # Move to device
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    acc_max = 0

    for epoch in range(1, 10+1):
      loss = train_epoch(model, train_loader, optimizer)
      acc = test_accuracy(model, test_loader)
      if acc > acc_max:
        acc_max = acc

      print(f"Batch Size: {i:2d} | DNN Size: {j:2d} | Epoch {epoch:2d} | Loss: {loss:.4f} | Test Acc: {acc:.4f}")

    torch.save(model.state_dict(), f"mnist_minivgg_{i}_{j}.pt")
    print(f"Model_{i}_{j} Finished Training with max accuracy: {acc_max}")
    acc_list.append(acc_max)


Batch Size: 32 | DNN Size: 32 | Epoch  1 | Loss: 0.1220 | Test Acc: 0.9878
Batch Size: 32 | DNN Size: 32 | Epoch  2 | Loss: 0.0434 | Test Acc: 0.9832
Batch Size: 32 | DNN Size: 32 | Epoch  3 | Loss: 0.0306 | Test Acc: 0.9902
Batch Size: 32 | DNN Size: 32 | Epoch  4 | Loss: 0.0234 | Test Acc: 0.9928
Batch Size: 32 | DNN Size: 32 | Epoch  5 | Loss: 0.0186 | Test Acc: 0.9914
Batch Size: 32 | DNN Size: 32 | Epoch  6 | Loss: 0.0147 | Test Acc: 0.9940
Batch Size: 32 | DNN Size: 32 | Epoch  7 | Loss: 0.0139 | Test Acc: 0.9931
Batch Size: 32 | DNN Size: 32 | Epoch  8 | Loss: 0.0102 | Test Acc: 0.9928
Batch Size: 32 | DNN Size: 32 | Epoch  9 | Loss: 0.0103 | Test Acc: 0.9914
Batch Size: 32 | DNN Size: 32 | Epoch 10 | Loss: 0.0088 | Test Acc: 0.9938
Model_32_32 Finished Training with max accuracy: 0.994
Batch Size: 32 | DNN Size: 64 | Epoch  1 | Loss: 0.1261 | Test Acc: 0.9839
Batch Size: 32 | DNN Size: 64 | Epoch  2 | Loss: 0.0430 | Test Acc: 0.9897
Batch Size: 32 | DNN Size: 64 | Epoch  3 | Lo

In [ ]:
#Below is the output for the different batch sizes(32-96) and Final DNN layer size(32-128).
#Test accuracy of the models were all similar in the low 99s.
#Highest is 32x96, lower batch size leads to better accuracy at the cost of compute, as SGD would be able to take advantage of more loss surfaces and escape local minima.
#The DNN size parameter did not seem to change performance, this model is already extremely accurate and its possible that this parameter was not tweaked enough to change its accuracy.
'''
Batch Size: 32 | DNN Size: 32 | Epoch  1 | Loss: 0.1220 | Test Acc: 0.9878
Batch Size: 32 | DNN Size: 32 | Epoch  2 | Loss: 0.0434 | Test Acc: 0.9832
Batch Size: 32 | DNN Size: 32 | Epoch  3 | Loss: 0.0306 | Test Acc: 0.9902
Batch Size: 32 | DNN Size: 32 | Epoch  4 | Loss: 0.0234 | Test Acc: 0.9928
Batch Size: 32 | DNN Size: 32 | Epoch  5 | Loss: 0.0186 | Test Acc: 0.9914
Batch Size: 32 | DNN Size: 32 | Epoch  6 | Loss: 0.0147 | Test Acc: 0.9940
Batch Size: 32 | DNN Size: 32 | Epoch  7 | Loss: 0.0139 | Test Acc: 0.9931
Batch Size: 32 | DNN Size: 32 | Epoch  8 | Loss: 0.0102 | Test Acc: 0.9928
Batch Size: 32 | DNN Size: 32 | Epoch  9 | Loss: 0.0103 | Test Acc: 0.9914
Batch Size: 32 | DNN Size: 32 | Epoch 10 | Loss: 0.0088 | Test Acc: 0.9938
Model_32_32 Finished Training with max accuracy: 0.994
Batch Size: 32 | DNN Size: 64 | Epoch  1 | Loss: 0.1261 | Test Acc: 0.9839
Batch Size: 32 | DNN Size: 64 | Epoch  2 | Loss: 0.0430 | Test Acc: 0.9897
Batch Size: 32 | DNN Size: 64 | Epoch  3 | Loss: 0.0296 | Test Acc: 0.9883
Batch Size: 32 | DNN Size: 64 | Epoch  4 | Loss: 0.0232 | Test Acc: 0.9891
Batch Size: 32 | DNN Size: 64 | Epoch  5 | Loss: 0.0189 | Test Acc: 0.9912
Batch Size: 32 | DNN Size: 64 | Epoch  6 | Loss: 0.0149 | Test Acc: 0.9915
Batch Size: 32 | DNN Size: 64 | Epoch  7 | Loss: 0.0110 | Test Acc: 0.9884
Batch Size: 32 | DNN Size: 64 | Epoch  8 | Loss: 0.0119 | Test Acc: 0.9924
Batch Size: 32 | DNN Size: 64 | Epoch  9 | Loss: 0.0097 | Test Acc: 0.9881
Batch Size: 32 | DNN Size: 64 | Epoch 10 | Loss: 0.0085 | Test Acc: 0.9937
Model_32_64 Finished Training with max accuracy: 0.9937
Batch Size: 32 | DNN Size: 96 | Epoch  1 | Loss: 0.1145 | Test Acc: 0.9861
Batch Size: 32 | DNN Size: 96 | Epoch  2 | Loss: 0.0396 | Test Acc: 0.9908
Batch Size: 32 | DNN Size: 96 | Epoch  3 | Loss: 0.0274 | Test Acc: 0.9917
Batch Size: 32 | DNN Size: 96 | Epoch  4 | Loss: 0.0211 | Test Acc: 0.9921
Batch Size: 32 | DNN Size: 96 | Epoch  5 | Loss: 0.0182 | Test Acc: 0.9917
Batch Size: 32 | DNN Size: 96 | Epoch  6 | Loss: 0.0136 | Test Acc: 0.9915
Batch Size: 32 | DNN Size: 96 | Epoch  7 | Loss: 0.0106 | Test Acc: 0.9934
Batch Size: 32 | DNN Size: 96 | Epoch  8 | Loss: 0.0113 | Test Acc: 0.9933
Batch Size: 32 | DNN Size: 96 | Epoch  9 | Loss: 0.0082 | Test Acc: 0.9947
Batch Size: 32 | DNN Size: 96 | Epoch 10 | Loss: 0.0083 | Test Acc: 0.9922
Model_32_96 Finished Training with max accuracy: 0.9947
Batch Size: 32 | DNN Size: 128 | Epoch  1 | Loss: 0.1142 | Test Acc: 0.9861
Batch Size: 32 | DNN Size: 128 | Epoch  2 | Loss: 0.0394 | Test Acc: 0.9909
Batch Size: 32 | DNN Size: 128 | Epoch  3 | Loss: 0.0279 | Test Acc: 0.9886
Batch Size: 32 | DNN Size: 128 | Epoch  4 | Loss: 0.0207 | Test Acc: 0.9865
Batch Size: 32 | DNN Size: 128 | Epoch  5 | Loss: 0.0153 | Test Acc: 0.9922
Batch Size: 32 | DNN Size: 128 | Epoch  6 | Loss: 0.0137 | Test Acc: 0.9926
Batch Size: 32 | DNN Size: 128 | Epoch  7 | Loss: 0.0103 | Test Acc: 0.9917
Batch Size: 32 | DNN Size: 128 | Epoch  8 | Loss: 0.0098 | Test Acc: 0.9902
Batch Size: 32 | DNN Size: 128 | Epoch  9 | Loss: 0.0087 | Test Acc: 0.9922
Batch Size: 32 | DNN Size: 128 | Epoch 10 | Loss: 0.0091 | Test Acc: 0.9920
Model_32_128 Finished Training with max accuracy: 0.9926
Batch Size: 48 | DNN Size: 32 | Epoch  1 | Loss: 0.1634 | Test Acc: 0.9805
Batch Size: 48 | DNN Size: 32 | Epoch  2 | Loss: 0.0481 | Test Acc: 0.9885
Batch Size: 48 | DNN Size: 32 | Epoch  3 | Loss: 0.0353 | Test Acc: 0.9917
Batch Size: 48 | DNN Size: 32 | Epoch  4 | Loss: 0.0264 | Test Acc: 0.9854
Batch Size: 48 | DNN Size: 32 | Epoch  5 | Loss: 0.0222 | Test Acc: 0.9913
Batch Size: 48 | DNN Size: 32 | Epoch  6 | Loss: 0.0188 | Test Acc: 0.9909
Batch Size: 48 | DNN Size: 32 | Epoch  7 | Loss: 0.0139 | Test Acc: 0.9903
Batch Size: 48 | DNN Size: 32 | Epoch  8 | Loss: 0.0127 | Test Acc: 0.9918
Batch Size: 48 | DNN Size: 32 | Epoch  9 | Loss: 0.0125 | Test Acc: 0.9871
Batch Size: 48 | DNN Size: 32 | Epoch 10 | Loss: 0.0092 | Test Acc: 0.9925
Model_48_32 Finished Training with max accuracy: 0.9925
Batch Size: 48 | DNN Size: 64 | Epoch  1 | Loss: 0.1285 | Test Acc: 0.9855
Batch Size: 48 | DNN Size: 64 | Epoch  2 | Loss: 0.0422 | Test Acc: 0.9881
Batch Size: 48 | DNN Size: 64 | Epoch  3 | Loss: 0.0300 | Test Acc: 0.9915
Batch Size: 48 | DNN Size: 64 | Epoch  4 | Loss: 0.0214 | Test Acc: 0.9883
Batch Size: 48 | DNN Size: 64 | Epoch  5 | Loss: 0.0187 | Test Acc: 0.9924
Batch Size: 48 | DNN Size: 64 | Epoch  6 | Loss: 0.0134 | Test Acc: 0.9931
Batch Size: 48 | DNN Size: 64 | Epoch  7 | Loss: 0.0127 | Test Acc: 0.9929
Batch Size: 48 | DNN Size: 64 | Epoch  8 | Loss: 0.0100 | Test Acc: 0.9921
Batch Size: 48 | DNN Size: 64 | Epoch  9 | Loss: 0.0099 | Test Acc: 0.9909
Batch Size: 48 | DNN Size: 64 | Epoch 10 | Loss: 0.0092 | Test Acc: 0.9917
Model_48_64 Finished Training with max accuracy: 0.9931
Batch Size: 48 | DNN Size: 96 | Epoch  1 | Loss: 0.1210 | Test Acc: 0.9853
Batch Size: 48 | DNN Size: 96 | Epoch  2 | Loss: 0.0387 | Test Acc: 0.9885
Batch Size: 48 | DNN Size: 96 | Epoch  3 | Loss: 0.0284 | Test Acc: 0.9898
Batch Size: 48 | DNN Size: 96 | Epoch  4 | Loss: 0.0207 | Test Acc: 0.9901
Batch Size: 48 | DNN Size: 96 | Epoch  5 | Loss: 0.0157 | Test Acc: 0.9921
Batch Size: 48 | DNN Size: 96 | Epoch  6 | Loss: 0.0134 | Test Acc: 0.9913
Batch Size: 48 | DNN Size: 96 | Epoch  7 | Loss: 0.0114 | Test Acc: 0.9925
Batch Size: 48 | DNN Size: 96 | Epoch  8 | Loss: 0.0090 | Test Acc: 0.9923
Batch Size: 48 | DNN Size: 96 | Epoch  9 | Loss: 0.0094 | Test Acc: 0.9898
Batch Size: 48 | DNN Size: 96 | Epoch 10 | Loss: 0.0082 | Test Acc: 0.9926
Model_48_96 Finished Training with max accuracy: 0.9926
Batch Size: 48 | DNN Size: 128 | Epoch  1 | Loss: 0.1255 | Test Acc: 0.9877
Batch Size: 48 | DNN Size: 128 | Epoch  2 | Loss: 0.0381 | Test Acc: 0.9892
Batch Size: 48 | DNN Size: 128 | Epoch  3 | Loss: 0.0260 | Test Acc: 0.9885
Batch Size: 48 | DNN Size: 128 | Epoch  4 | Loss: 0.0192 | Test Acc: 0.9908
Batch Size: 48 | DNN Size: 128 | Epoch  5 | Loss: 0.0173 | Test Acc: 0.9923
Batch Size: 48 | DNN Size: 128 | Epoch  6 | Loss: 0.0128 | Test Acc: 0.9923
Batch Size: 48 | DNN Size: 128 | Epoch  7 | Loss: 0.0120 | Test Acc: 0.9928
Batch Size: 48 | DNN Size: 128 | Epoch  8 | Loss: 0.0090 | Test Acc: 0.9920
Batch Size: 48 | DNN Size: 128 | Epoch  9 | Loss: 0.0072 | Test Acc: 0.9919
Batch Size: 48 | DNN Size: 128 | Epoch 10 | Loss: 0.0079 | Test Acc: 0.9895
Model_48_128 Finished Training with max accuracy: 0.9928
Batch Size: 64 | DNN Size: 32 | Epoch  1 | Loss: 0.1585 | Test Acc: 0.9886
Batch Size: 64 | DNN Size: 32 | Epoch  2 | Loss: 0.0424 | Test Acc: 0.9872
Batch Size: 64 | DNN Size: 32 | Epoch  3 | Loss: 0.0314 | Test Acc: 0.9895
Batch Size: 64 | DNN Size: 32 | Epoch  4 | Loss: 0.0219 | Test Acc: 0.9927
Batch Size: 64 | DNN Size: 32 | Epoch  5 | Loss: 0.0187 | Test Acc: 0.9915
Batch Size: 64 | DNN Size: 32 | Epoch  6 | Loss: 0.0142 | Test Acc: 0.9924
Batch Size: 64 | DNN Size: 32 | Epoch  7 | Loss: 0.0129 | Test Acc: 0.9908
Batch Size: 64 | DNN Size: 32 | Epoch  8 | Loss: 0.0104 | Test Acc: 0.9920
Batch Size: 64 | DNN Size: 32 | Epoch  9 | Loss: 0.0106 | Test Acc: 0.9922
Batch Size: 64 | DNN Size: 32 | Epoch 10 | Loss: 0.0082 | Test Acc: 0.9915
Model_64_32 Finished Training with max accuracy: 0.9927
Batch Size: 64 | DNN Size: 64 | Epoch  1 | Loss: 0.1441 | Test Acc: 0.9871
Batch Size: 64 | DNN Size: 64 | Epoch  2 | Loss: 0.0421 | Test Acc: 0.9904
Batch Size: 64 | DNN Size: 64 | Epoch  3 | Loss: 0.0297 | Test Acc: 0.9905
Batch Size: 64 | DNN Size: 64 | Epoch  4 | Loss: 0.0226 | Test Acc: 0.9914
Batch Size: 64 | DNN Size: 64 | Epoch  5 | Loss: 0.0189 | Test Acc: 0.9902
Batch Size: 64 | DNN Size: 64 | Epoch  6 | Loss: 0.0149 | Test Acc: 0.9898
Batch Size: 64 | DNN Size: 64 | Epoch  7 | Loss: 0.0127 | Test Acc: 0.9916
Batch Size: 64 | DNN Size: 64 | Epoch  8 | Loss: 0.0102 | Test Acc: 0.9928
Batch Size: 64 | DNN Size: 64 | Epoch  9 | Loss: 0.0080 | Test Acc: 0.9922
Batch Size: 64 | DNN Size: 64 | Epoch 10 | Loss: 0.0101 | Test Acc: 0.9914
Model_64_64 Finished Training with max accuracy: 0.9928
Batch Size: 64 | DNN Size: 96 | Epoch  1 | Loss: 0.1277 | Test Acc: 0.9809
Batch Size: 64 | DNN Size: 96 | Epoch  2 | Loss: 0.0401 | Test Acc: 0.9896
Batch Size: 64 | DNN Size: 96 | Epoch  3 | Loss: 0.0276 | Test Acc: 0.9921
Batch Size: 64 | DNN Size: 96 | Epoch  4 | Loss: 0.0211 | Test Acc: 0.9880
Batch Size: 64 | DNN Size: 96 | Epoch  5 | Loss: 0.0174 | Test Acc: 0.9915
Batch Size: 64 | DNN Size: 96 | Epoch  6 | Loss: 0.0150 | Test Acc: 0.9896
Batch Size: 64 | DNN Size: 96 | Epoch  7 | Loss: 0.0112 | Test Acc: 0.9903
Batch Size: 64 | DNN Size: 96 | Epoch  8 | Loss: 0.0109 | Test Acc: 0.9912
Batch Size: 64 | DNN Size: 96 | Epoch  9 | Loss: 0.0090 | Test Acc: 0.9917
Batch Size: 64 | DNN Size: 96 | Epoch 10 | Loss: 0.0085 | Test Acc: 0.9922
Model_64_96 Finished Training with max accuracy: 0.9922
Batch Size: 64 | DNN Size: 128 | Epoch  1 | Loss: 0.1294 | Test Acc: 0.9780
Batch Size: 64 | DNN Size: 128 | Epoch  2 | Loss: 0.0401 | Test Acc: 0.9910
Batch Size: 64 | DNN Size: 128 | Epoch  3 | Loss: 0.0271 | Test Acc: 0.9898
Batch Size: 64 | DNN Size: 128 | Epoch  4 | Loss: 0.0211 | Test Acc: 0.9871
Batch Size: 64 | DNN Size: 128 | Epoch  5 | Loss: 0.0176 | Test Acc: 0.9920
Batch Size: 64 | DNN Size: 128 | Epoch  6 | Loss: 0.0145 | Test Acc: 0.9928
Batch Size: 64 | DNN Size: 128 | Epoch  7 | Loss: 0.0118 | Test Acc: 0.9890
Batch Size: 64 | DNN Size: 128 | Epoch  8 | Loss: 0.0099 | Test Acc: 0.9930
Batch Size: 64 | DNN Size: 128 | Epoch  9 | Loss: 0.0068 | Test Acc: 0.9915
Batch Size: 64 | DNN Size: 128 | Epoch 10 | Loss: 0.0087 | Test Acc: 0.9901
Model_64_128 Finished Training with max accuracy: 0.993
Batch Size: 80 | DNN Size: 32 | Epoch  1 | Loss: 0.1608 | Test Acc: 0.9863
Batch Size: 80 | DNN Size: 32 | Epoch  2 | Loss: 0.0437 | Test Acc: 0.9894
Batch Size: 80 | DNN Size: 32 | Epoch  3 | Loss: 0.0315 | Test Acc: 0.9887
Batch Size: 80 | DNN Size: 32 | Epoch  4 | Loss: 0.0221 | Test Acc: 0.9883
Batch Size: 80 | DNN Size: 32 | Epoch  5 | Loss: 0.0198 | Test Acc: 0.9922
Batch Size: 80 | DNN Size: 32 | Epoch  6 | Loss: 0.0164 | Test Acc: 0.9919
Batch Size: 80 | DNN Size: 32 | Epoch  7 | Loss: 0.0143 | Test Acc: 0.9901
Batch Size: 80 | DNN Size: 32 | Epoch  8 | Loss: 0.0112 | Test Acc: 0.9897
Batch Size: 80 | DNN Size: 32 | Epoch  9 | Loss: 0.0107 | Test Acc: 0.9907
Batch Size: 80 | DNN Size: 32 | Epoch 10 | Loss: 0.0077 | Test Acc: 0.9921
Model_80_32 Finished Training with max accuracy: 0.9922
Batch Size: 80 | DNN Size: 64 | Epoch  1 | Loss: 0.1542 | Test Acc: 0.9861
Batch Size: 80 | DNN Size: 64 | Epoch  2 | Loss: 0.0445 | Test Acc: 0.9877
Batch Size: 80 | DNN Size: 64 | Epoch  3 | Loss: 0.0302 | Test Acc: 0.9900
Batch Size: 80 | DNN Size: 64 | Epoch  4 | Loss: 0.0229 | Test Acc: 0.9896
Batch Size: 80 | DNN Size: 64 | Epoch  5 | Loss: 0.0187 | Test Acc: 0.9905
Batch Size: 80 | DNN Size: 64 | Epoch  6 | Loss: 0.0169 | Test Acc: 0.9900
Batch Size: 80 | DNN Size: 64 | Epoch  7 | Loss: 0.0129 | Test Acc: 0.9913
Batch Size: 80 | DNN Size: 64 | Epoch  8 | Loss: 0.0094 | Test Acc: 0.9911
Batch Size: 80 | DNN Size: 64 | Epoch  9 | Loss: 0.0115 | Test Acc: 0.9923
Batch Size: 80 | DNN Size: 64 | Epoch 10 | Loss: 0.0086 | Test Acc: 0.9924
Model_80_64 Finished Training with max accuracy: 0.9924
Batch Size: 80 | DNN Size: 96 | Epoch  1 | Loss: 0.1431 | Test Acc: 0.9875
Batch Size: 80 | DNN Size: 96 | Epoch  2 | Loss: 0.0405 | Test Acc: 0.9867
Batch Size: 80 | DNN Size: 96 | Epoch  3 | Loss: 0.0274 | Test Acc: 0.9920
Batch Size: 80 | DNN Size: 96 | Epoch  4 | Loss: 0.0216 | Test Acc: 0.9905
Batch Size: 80 | DNN Size: 96 | Epoch  5 | Loss: 0.0157 | Test Acc: 0.9922
Batch Size: 80 | DNN Size: 96 | Epoch  6 | Loss: 0.0140 | Test Acc: 0.9928
Batch Size: 80 | DNN Size: 96 | Epoch  7 | Loss: 0.0110 | Test Acc: 0.9932
Batch Size: 80 | DNN Size: 96 | Epoch  8 | Loss: 0.0102 | Test Acc: 0.9928
Batch Size: 80 | DNN Size: 96 | Epoch  9 | Loss: 0.0079 | Test Acc: 0.9909
Batch Size: 80 | DNN Size: 96 | Epoch 10 | Loss: 0.0069 | Test Acc: 0.9920
Model_80_96 Finished Training with max accuracy: 0.9932
Batch Size: 80 | DNN Size: 128 | Epoch  1 | Loss: 0.1381 | Test Acc: 0.9879
Batch Size: 80 | DNN Size: 128 | Epoch  2 | Loss: 0.0409 | Test Acc: 0.9879
Batch Size: 80 | DNN Size: 128 | Epoch  3 | Loss: 0.0280 | Test Acc: 0.9896
Batch Size: 80 | DNN Size: 128 | Epoch  4 | Loss: 0.0207 | Test Acc: 0.9922
Batch Size: 80 | DNN Size: 128 | Epoch  5 | Loss: 0.0167 | Test Acc: 0.9925
Batch Size: 80 | DNN Size: 128 | Epoch  6 | Loss: 0.0118 | Test Acc: 0.9929
Batch Size: 80 | DNN Size: 128 | Epoch  7 | Loss: 0.0126 | Test Acc: 0.9913
Batch Size: 80 | DNN Size: 128 | Epoch  8 | Loss: 0.0095 | Test Acc: 0.9922
Batch Size: 80 | DNN Size: 128 | Epoch  9 | Loss: 0.0069 | Test Acc: 0.9917
Batch Size: 80 | DNN Size: 128 | Epoch 10 | Loss: 0.0088 | Test Acc: 0.9899
Model_80_128 Finished Training with max accuracy: 0.9929
Batch Size: 96 | DNN Size: 32 | Epoch  1 | Loss: 0.1764 | Test Acc: 0.9846
Batch Size: 96 | DNN Size: 32 | Epoch  2 | Loss: 0.0487 | Test Acc: 0.9869
Batch Size: 96 | DNN Size: 32 | Epoch  3 | Loss: 0.0338 | Test Acc: 0.9905
Batch Size: 96 | DNN Size: 32 | Epoch  4 | Loss: 0.0248 | Test Acc: 0.9909
Batch Size: 96 | DNN Size: 32 | Epoch  5 | Loss: 0.0194 | Test Acc: 0.9906
Batch Size: 96 | DNN Size: 32 | Epoch  6 | Loss: 0.0181 | Test Acc: 0.9904
Batch Size: 96 | DNN Size: 32 | Epoch  7 | Loss: 0.0154 | Test Acc: 0.9938
Batch Size: 96 | DNN Size: 32 | Epoch  8 | Loss: 0.0122 | Test Acc: 0.9903
Batch Size: 96 | DNN Size: 32 | Epoch  9 | Loss: 0.0104 | Test Acc: 0.9933
Batch Size: 96 | DNN Size: 32 | Epoch 10 | Loss: 0.0094 | Test Acc: 0.9917
Model_96_32 Finished Training with max accuracy: 0.9938
Batch Size: 96 | DNN Size: 64 | Epoch  1 | Loss: 0.1536 | Test Acc: 0.9828
Batch Size: 96 | DNN Size: 64 | Epoch  2 | Loss: 0.0444 | Test Acc: 0.9902
Batch Size: 96 | DNN Size: 64 | Epoch  3 | Loss: 0.0306 | Test Acc: 0.9900
Batch Size: 96 | DNN Size: 64 | Epoch  4 | Loss: 0.0232 | Test Acc: 0.9893
Batch Size: 96 | DNN Size: 64 | Epoch  5 | Loss: 0.0177 | Test Acc: 0.9919
Batch Size: 96 | DNN Size: 64 | Epoch  6 | Loss: 0.0138 | Test Acc: 0.9927
Batch Size: 96 | DNN Size: 64 | Epoch  7 | Loss: 0.0129 | Test Acc: 0.9922
Batch Size: 96 | DNN Size: 64 | Epoch  8 | Loss: 0.0097 | Test Acc: 0.9906
Batch Size: 96 | DNN Size: 64 | Epoch  9 | Loss: 0.0091 | Test Acc: 0.9921
Batch Size: 96 | DNN Size: 64 | Epoch 10 | Loss: 0.0093 | Test Acc: 0.9896
Model_96_64 Finished Training with max accuracy: 0.9927
Batch Size: 96 | DNN Size: 96 | Epoch  1 | Loss: 0.1511 | Test Acc: 0.9879
Batch Size: 96 | DNN Size: 96 | Epoch  2 | Loss: 0.0403 | Test Acc: 0.9880
Batch Size: 96 | DNN Size: 96 | Epoch  3 | Loss: 0.0279 | Test Acc: 0.9895
Batch Size: 96 | DNN Size: 96 | Epoch  4 | Loss: 0.0207 | Test Acc: 0.9935
Batch Size: 96 | DNN Size: 96 | Epoch  5 | Loss: 0.0157 | Test Acc: 0.9912
Batch Size: 96 | DNN Size: 96 | Epoch  6 | Loss: 0.0131 | Test Acc: 0.9894
Batch Size: 96 | DNN Size: 96 | Epoch  7 | Loss: 0.0116 | Test Acc: 0.9921
Batch Size: 96 | DNN Size: 96 | Epoch  8 | Loss: 0.0105 | Test Acc: 0.9916
Batch Size: 96 | DNN Size: 96 | Epoch  9 | Loss: 0.0082 | Test Acc: 0.9917
Batch Size: 96 | DNN Size: 96 | Epoch 10 | Loss: 0.0085 | Test Acc: 0.9919
Model_96_96 Finished Training with max accuracy: 0.9935
Batch Size: 96 | DNN Size: 128 | Epoch  1 | Loss: 0.1471 | Test Acc: 0.9861
Batch Size: 96 | DNN Size: 128 | Epoch  2 | Loss: 0.0401 | Test Acc: 0.9899
Batch Size: 96 | DNN Size: 128 | Epoch  3 | Loss: 0.0254 | Test Acc: 0.9921
Batch Size: 96 | DNN Size: 128 | Epoch  4 | Loss: 0.0200 | Test Acc: 0.9913
Batch Size: 96 | DNN Size: 128 | Epoch  5 | Loss: 0.0164 | Test Acc: 0.9914
Batch Size: 96 | DNN Size: 128 | Epoch  6 | Loss: 0.0140 | Test Acc: 0.9908
Batch Size: 96 | DNN Size: 128 | Epoch  7 | Loss: 0.0100 | Test Acc: 0.9925
Batch Size: 96 | DNN Size: 128 | Epoch  8 | Loss: 0.0091 | Test Acc: 0.9922
Batch Size: 96 | DNN Size: 128 | Epoch  9 | Loss: 0.0080 | Test Acc: 0.9920
Batch Size: 96 | DNN Size: 128 | Epoch 10 | Loss: 0.0084 | Test Acc: 0.9917
Model_96_128 Finished Training with max accuracy: 0.9925
CPU times: user 1h 12min 18s, sys: 18.1 s, total: 1h 12min 36s
Wall time: 1h 13min 7s'''

In [9]:
shuffled_model = MNISTCNN().to(device)
shuffled_optimizer = optim.SGD(shuffled_model.parameters(), lr=.01, momentum=.9)
criterion = nn.CrossEntropyLoss()

In [10]:
%%time
#Question 4 MNUTS:
epochs = 100

for epoch in range(1, epochs + 1):
    loss = train_epoch(shuffled_model, shuffled_train_loader, shuffled_optimizer)
    acc = test_accuracy(shuffled_model, shuffled_train_loader)

    print(f"Epoch {epoch:2d} | Loss: {loss:.4f} | Test Acc: {acc:.4f}")

torch.save(shuffled_model.state_dict(), "mnist_minivgg_shuffled.pt")

Epoch  1 | Loss: 2.3017 | Test Acc: 0.1124
Epoch  2 | Loss: 2.3013 | Test Acc: 0.1124
Epoch  3 | Loss: 2.3012 | Test Acc: 0.1124
Epoch  4 | Loss: 2.3011 | Test Acc: 0.1124
Epoch  5 | Loss: 2.3010 | Test Acc: 0.1127
Epoch  6 | Loss: 2.3008 | Test Acc: 0.1124
Epoch  7 | Loss: 2.3008 | Test Acc: 0.1124
Epoch  8 | Loss: 2.3006 | Test Acc: 0.1124
Epoch  9 | Loss: 2.3004 | Test Acc: 0.1124
Epoch 10 | Loss: 2.3002 | Test Acc: 0.1128
Epoch 11 | Loss: 2.3000 | Test Acc: 0.1135
Epoch 12 | Loss: 2.2997 | Test Acc: 0.1148
Epoch 13 | Loss: 2.2995 | Test Acc: 0.1141
Epoch 14 | Loss: 2.2991 | Test Acc: 0.1183
Epoch 15 | Loss: 2.2987 | Test Acc: 0.1168
Epoch 16 | Loss: 2.2981 | Test Acc: 0.1169
Epoch 17 | Loss: 2.2979 | Test Acc: 0.1189
Epoch 18 | Loss: 2.2973 | Test Acc: 0.1200
Epoch 19 | Loss: 2.2966 | Test Acc: 0.1224
Epoch 20 | Loss: 2.2957 | Test Acc: 0.1233
Epoch 21 | Loss: 2.2950 | Test Acc: 0.1276
Epoch 22 | Loss: 2.2942 | Test Acc: 0.1250
Epoch 23 | Loss: 2.2927 | Test Acc: 0.1309
Epoch 24 | 

In [ ]:
#Initially stuck at .1124, the proportion of the most common label.
#I hiked up batch size and learning rate, and swapped to SGD and the model began to overfit, final accuracy=~93%

In [ ]:
print(torch.bincount(shuffled_targets)/60000.)

tensor([0.0987, 0.1124, 0.0993, 0.1022, 0.0974, 0.0904, 0.0986, 0.1044, 0.0975,
        0.0992])


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    #transforms.RandomRotation(10),
    #transforms.RandomAffine(0, translate=(0.1,0.1))
])

BNDO_train = datasets.MNIST("./data", train=True, download=True, transform=transform)
BNDO_test  = datasets.MNIST("./data", train=False, download=True, transform=transform)

BNDO_train_loader = DataLoader(BNDO_train, batch_size=64, shuffle=True)
BNDO_test_loader  = DataLoader(BNDO_test, batch_size=1000)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1),
        nn.BatchNorm2d(32), #
        nn.ReLU(),
        nn.Conv2d(32, 32, 3, padding=1),
        nn.BatchNorm2d(32), #
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout(0.25), #

        nn.Conv2d(32, 64, 3, padding=1),
        nn.BatchNorm2d(64), #
        nn.ReLU(),
        nn.Conv2d(64, 64, 3, padding=1),
        nn.BatchNorm2d(64), #
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout(0.25), #

        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(0.5), #
        nn.Linear(128, 10)
)

    def forward(self, x):
        return self.net(x)

In [ ]:
BNDO_model = CNN().to(device)
opt = optim.Adam(BNDO_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


In [ ]:
BNDO_model_list = []
for i in range(20):
    testmodel = CNN().to(device)
    testmodel.net = nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1),
        nn.BatchNorm2d(32), #
        nn.ReLU(),
        nn.Conv2d(32, 32, 3, padding=1),
        nn.BatchNorm2d(32), #
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout(i/40.), #

        nn.Conv2d(32, 64, 3, padding=1),
        nn.BatchNorm2d(64), #
        nn.ReLU(),
        nn.Conv2d(64, 64, 3, padding=1),
        nn.BatchNorm2d(64), #
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Dropout(i/40.), #

        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(0.5), #
        nn.Linear(128, 10)
        )
    opt = optim.Adam(testmodel.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    BNDO_model_list.append(testmodel)

In [ ]:
#Question 5 HyperParameters testing:
for index, value in enumerate(BNDO_model_list):
  value.to(device) # Ensure the model is on the correct device
  opt = optim.Adam(value.parameters(), lr=1e-3) # Create a new optimizer for the current model
  loss_fn = nn.CrossEntropyLoss() # Re-instantiate loss_fn (or ensure it's defined globally once for device if needed)

  for epoch in range(15):
      value.train()
      for x, y in BNDO_train_loader:
          x, y = x.to(device), y.to(device)
          opt.zero_grad()
          loss = loss_fn(value(x), y)
          loss.backward()
          opt.step()

      # evaluate
      value.eval()
      correct = 0
      with torch.no_grad():
          for x, y in BNDO_test_loader:
              x, y = x.to(device), y.to(device)
              correct += (value(x).argmax(1) == y).sum().item()

      print(f"Dropout Rate: {index/40.:.3f} Epoch {epoch+1}: accuracy = {correct/len(BNDO_test):.4f}")
  print(f"Model {index+1} finished training")
  torch.save(value.state_dict(), f"mnist_minivgg_BNDO_{index}.pt")

Dropout Rate: 0.000 Epoch 1: accuracy = 0.9860
Dropout Rate: 0.000 Epoch 2: accuracy = 0.9915
Dropout Rate: 0.000 Epoch 3: accuracy = 0.9877
Dropout Rate: 0.000 Epoch 4: accuracy = 0.9923
Dropout Rate: 0.000 Epoch 5: accuracy = 0.9932
Dropout Rate: 0.000 Epoch 6: accuracy = 0.9933
Dropout Rate: 0.000 Epoch 7: accuracy = 0.9950
Dropout Rate: 0.000 Epoch 8: accuracy = 0.9906
Dropout Rate: 0.000 Epoch 9: accuracy = 0.9942
Dropout Rate: 0.000 Epoch 10: accuracy = 0.9938
Dropout Rate: 0.000 Epoch 11: accuracy = 0.9928
Dropout Rate: 0.000 Epoch 12: accuracy = 0.9939
Dropout Rate: 0.000 Epoch 13: accuracy = 0.9942
Dropout Rate: 0.000 Epoch 14: accuracy = 0.9952
Dropout Rate: 0.000 Epoch 15: accuracy = 0.9957
Model 1 finished training
Dropout Rate: 0.025 Epoch 1: accuracy = 0.9891
Dropout Rate: 0.025 Epoch 2: accuracy = 0.9914
Dropout Rate: 0.025 Epoch 3: accuracy = 0.9922
Dropout Rate: 0.025 Epoch 4: accuracy = 0.9919
Dropout Rate: 0.025 Epoch 5: accuracy = 0.9909
Dropout Rate: 0.025 Epoch 6:

In [ ]:
#I trained the MiniVGG model and varied the dropout rate of the convolutional layers.
#The highest test accuracy belonged to Model 19 with dropout rate .450.
#Overall there was little variation in the model accuracy among the dropout rates.
#We could test the robustness of the models by perturbing the images slightly and seeing which ones perform better.
#We expect that the models with higher dropout rates perform better.

'''
Dropout Rate: 0.000 Epoch 1: accuracy = 0.9860
Dropout Rate: 0.000 Epoch 2: accuracy = 0.9915
Dropout Rate: 0.000 Epoch 3: accuracy = 0.9877
Dropout Rate: 0.000 Epoch 4: accuracy = 0.9923
Dropout Rate: 0.000 Epoch 5: accuracy = 0.9932
Dropout Rate: 0.000 Epoch 6: accuracy = 0.9933
Dropout Rate: 0.000 Epoch 7: accuracy = 0.9950
Dropout Rate: 0.000 Epoch 8: accuracy = 0.9906
Dropout Rate: 0.000 Epoch 9: accuracy = 0.9942
Dropout Rate: 0.000 Epoch 10: accuracy = 0.9938
Dropout Rate: 0.000 Epoch 11: accuracy = 0.9928
Dropout Rate: 0.000 Epoch 12: accuracy = 0.9939
Dropout Rate: 0.000 Epoch 13: accuracy = 0.9942
Dropout Rate: 0.000 Epoch 14: accuracy = 0.9952
Dropout Rate: 0.000 Epoch 15: accuracy = 0.9957
Model 1 finished training
Dropout Rate: 0.025 Epoch 1: accuracy = 0.9891
Dropout Rate: 0.025 Epoch 2: accuracy = 0.9914
Dropout Rate: 0.025 Epoch 3: accuracy = 0.9922
Dropout Rate: 0.025 Epoch 4: accuracy = 0.9919
Dropout Rate: 0.025 Epoch 5: accuracy = 0.9909
Dropout Rate: 0.025 Epoch 6: accuracy = 0.9926
Dropout Rate: 0.025 Epoch 7: accuracy = 0.9943
Dropout Rate: 0.025 Epoch 8: accuracy = 0.9933
Dropout Rate: 0.025 Epoch 9: accuracy = 0.9942
Dropout Rate: 0.025 Epoch 10: accuracy = 0.9949
Dropout Rate: 0.025 Epoch 11: accuracy = 0.9951
Dropout Rate: 0.025 Epoch 12: accuracy = 0.9940
Dropout Rate: 0.025 Epoch 13: accuracy = 0.9955
Dropout Rate: 0.025 Epoch 14: accuracy = 0.9953
Dropout Rate: 0.025 Epoch 15: accuracy = 0.9949
Model 2 finished training
Dropout Rate: 0.050 Epoch 1: accuracy = 0.9894
Dropout Rate: 0.050 Epoch 2: accuracy = 0.9892
Dropout Rate: 0.050 Epoch 3: accuracy = 0.9918
Dropout Rate: 0.050 Epoch 4: accuracy = 0.9913
Dropout Rate: 0.050 Epoch 5: accuracy = 0.9924
Dropout Rate: 0.050 Epoch 6: accuracy = 0.9946
Dropout Rate: 0.050 Epoch 7: accuracy = 0.9938
Dropout Rate: 0.050 Epoch 8: accuracy = 0.9923
Dropout Rate: 0.050 Epoch 9: accuracy = 0.9939
Dropout Rate: 0.050 Epoch 10: accuracy = 0.9943
Dropout Rate: 0.050 Epoch 11: accuracy = 0.9947
Dropout Rate: 0.050 Epoch 12: accuracy = 0.9939
Dropout Rate: 0.050 Epoch 13: accuracy = 0.9946
Dropout Rate: 0.050 Epoch 14: accuracy = 0.9948
Dropout Rate: 0.050 Epoch 15: accuracy = 0.9951
Model 3 finished training
Dropout Rate: 0.075 Epoch 1: accuracy = 0.9893
Dropout Rate: 0.075 Epoch 2: accuracy = 0.9913
Dropout Rate: 0.075 Epoch 3: accuracy = 0.9920
Dropout Rate: 0.075 Epoch 4: accuracy = 0.9933
Dropout Rate: 0.075 Epoch 5: accuracy = 0.9937
Dropout Rate: 0.075 Epoch 6: accuracy = 0.9932
Dropout Rate: 0.075 Epoch 7: accuracy = 0.9943
Dropout Rate: 0.075 Epoch 8: accuracy = 0.9933
Dropout Rate: 0.075 Epoch 9: accuracy = 0.9955
Dropout Rate: 0.075 Epoch 10: accuracy = 0.9944
Dropout Rate: 0.075 Epoch 11: accuracy = 0.9942
Dropout Rate: 0.075 Epoch 12: accuracy = 0.9948
Dropout Rate: 0.075 Epoch 13: accuracy = 0.9939
Dropout Rate: 0.075 Epoch 14: accuracy = 0.9955
Dropout Rate: 0.075 Epoch 15: accuracy = 0.9955
Model 4 finished training
Dropout Rate: 0.100 Epoch 1: accuracy = 0.9694
Dropout Rate: 0.100 Epoch 2: accuracy = 0.9915
Dropout Rate: 0.100 Epoch 3: accuracy = 0.9917
Dropout Rate: 0.100 Epoch 4: accuracy = 0.9935
Dropout Rate: 0.100 Epoch 5: accuracy = 0.9937
Dropout Rate: 0.100 Epoch 6: accuracy = 0.9912
Dropout Rate: 0.100 Epoch 7: accuracy = 0.9950
Dropout Rate: 0.100 Epoch 8: accuracy = 0.9923
Dropout Rate: 0.100 Epoch 9: accuracy = 0.9951
Dropout Rate: 0.100 Epoch 10: accuracy = 0.9936
Dropout Rate: 0.100 Epoch 11: accuracy = 0.9936
Dropout Rate: 0.100 Epoch 12: accuracy = 0.9944
Dropout Rate: 0.100 Epoch 13: accuracy = 0.9944
Dropout Rate: 0.100 Epoch 14: accuracy = 0.9946
Dropout Rate: 0.100 Epoch 15: accuracy = 0.9939
Model 5 finished training
Dropout Rate: 0.125 Epoch 1: accuracy = 0.9844
Dropout Rate: 0.125 Epoch 2: accuracy = 0.9907
Dropout Rate: 0.125 Epoch 3: accuracy = 0.9904
Dropout Rate: 0.125 Epoch 4: accuracy = 0.9909
Dropout Rate: 0.125 Epoch 5: accuracy = 0.9934
Dropout Rate: 0.125 Epoch 6: accuracy = 0.9945
Dropout Rate: 0.125 Epoch 7: accuracy = 0.9942
Dropout Rate: 0.125 Epoch 8: accuracy = 0.9947
Dropout Rate: 0.125 Epoch 9: accuracy = 0.9938
Dropout Rate: 0.125 Epoch 10: accuracy = 0.9937
Dropout Rate: 0.125 Epoch 11: accuracy = 0.9944
Dropout Rate: 0.125 Epoch 12: accuracy = 0.9935
Dropout Rate: 0.125 Epoch 13: accuracy = 0.9955
Dropout Rate: 0.125 Epoch 14: accuracy = 0.9957
Dropout Rate: 0.125 Epoch 15: accuracy = 0.9951
Model 6 finished training
Dropout Rate: 0.150 Epoch 1: accuracy = 0.9777
Dropout Rate: 0.150 Epoch 2: accuracy = 0.9914
Dropout Rate: 0.150 Epoch 3: accuracy = 0.9924
Dropout Rate: 0.150 Epoch 4: accuracy = 0.9933
Dropout Rate: 0.150 Epoch 5: accuracy = 0.9925
Dropout Rate: 0.150 Epoch 6: accuracy = 0.9941
Dropout Rate: 0.150 Epoch 7: accuracy = 0.9931
Dropout Rate: 0.150 Epoch 8: accuracy = 0.9949
Dropout Rate: 0.150 Epoch 9: accuracy = 0.9931
Dropout Rate: 0.150 Epoch 10: accuracy = 0.9936
Dropout Rate: 0.150 Epoch 11: accuracy = 0.9944
Dropout Rate: 0.150 Epoch 12: accuracy = 0.9943
Dropout Rate: 0.150 Epoch 13: accuracy = 0.9951
Dropout Rate: 0.150 Epoch 14: accuracy = 0.9947
Dropout Rate: 0.150 Epoch 15: accuracy = 0.9950
Model 7 finished training
Dropout Rate: 0.175 Epoch 1: accuracy = 0.9837
Dropout Rate: 0.175 Epoch 2: accuracy = 0.9894
Dropout Rate: 0.175 Epoch 3: accuracy = 0.9887
Dropout Rate: 0.175 Epoch 4: accuracy = 0.9894
Dropout Rate: 0.175 Epoch 5: accuracy = 0.9929
Dropout Rate: 0.175 Epoch 6: accuracy = 0.9928
Dropout Rate: 0.175 Epoch 7: accuracy = 0.9932
Dropout Rate: 0.175 Epoch 8: accuracy = 0.9932
Dropout Rate: 0.175 Epoch 9: accuracy = 0.9950
Dropout Rate: 0.175 Epoch 10: accuracy = 0.9950
Dropout Rate: 0.175 Epoch 11: accuracy = 0.9928
Dropout Rate: 0.175 Epoch 12: accuracy = 0.9941
Dropout Rate: 0.175 Epoch 13: accuracy = 0.9950
Dropout Rate: 0.175 Epoch 14: accuracy = 0.9949
Dropout Rate: 0.175 Epoch 15: accuracy = 0.9950
Model 8 finished training
Dropout Rate: 0.200 Epoch 1: accuracy = 0.9852
Dropout Rate: 0.200 Epoch 2: accuracy = 0.9914
Dropout Rate: 0.200 Epoch 3: accuracy = 0.9818
Dropout Rate: 0.200 Epoch 4: accuracy = 0.9940
Dropout Rate: 0.200 Epoch 5: accuracy = 0.9934
Dropout Rate: 0.200 Epoch 6: accuracy = 0.9918
Dropout Rate: 0.200 Epoch 7: accuracy = 0.9941
Dropout Rate: 0.200 Epoch 8: accuracy = 0.9947
Dropout Rate: 0.200 Epoch 9: accuracy = 0.9946
Dropout Rate: 0.200 Epoch 10: accuracy = 0.9951
Dropout Rate: 0.200 Epoch 11: accuracy = 0.9946
Dropout Rate: 0.200 Epoch 12: accuracy = 0.9944
Dropout Rate: 0.200 Epoch 13: accuracy = 0.9948
Dropout Rate: 0.200 Epoch 14: accuracy = 0.9948
Dropout Rate: 0.200 Epoch 15: accuracy = 0.9952
Model 9 finished training
Dropout Rate: 0.225 Epoch 1: accuracy = 0.9878
Dropout Rate: 0.225 Epoch 2: accuracy = 0.9892
Dropout Rate: 0.225 Epoch 3: accuracy = 0.9912
Dropout Rate: 0.225 Epoch 4: accuracy = 0.9933
Dropout Rate: 0.225 Epoch 5: accuracy = 0.9924
Dropout Rate: 0.225 Epoch 6: accuracy = 0.9929
Dropout Rate: 0.225 Epoch 7: accuracy = 0.9923
Dropout Rate: 0.225 Epoch 8: accuracy = 0.9935
Dropout Rate: 0.225 Epoch 9: accuracy = 0.9939
Dropout Rate: 0.225 Epoch 10: accuracy = 0.9948
Dropout Rate: 0.225 Epoch 11: accuracy = 0.9935
Dropout Rate: 0.225 Epoch 12: accuracy = 0.9943
Dropout Rate: 0.225 Epoch 13: accuracy = 0.9953
Dropout Rate: 0.225 Epoch 14: accuracy = 0.9955
Dropout Rate: 0.225 Epoch 15: accuracy = 0.9949
Model 10 finished training
Dropout Rate: 0.250 Epoch 1: accuracy = 0.9838
Dropout Rate: 0.250 Epoch 2: accuracy = 0.9885
Dropout Rate: 0.250 Epoch 3: accuracy = 0.9919
Dropout Rate: 0.250 Epoch 4: accuracy = 0.9907
Dropout Rate: 0.250 Epoch 5: accuracy = 0.9931
Dropout Rate: 0.250 Epoch 6: accuracy = 0.9937
Dropout Rate: 0.250 Epoch 7: accuracy = 0.9925
Dropout Rate: 0.250 Epoch 8: accuracy = 0.9939
Dropout Rate: 0.250 Epoch 9: accuracy = 0.9936
Dropout Rate: 0.250 Epoch 10: accuracy = 0.9945
Dropout Rate: 0.250 Epoch 11: accuracy = 0.9946
Dropout Rate: 0.250 Epoch 12: accuracy = 0.9949
Dropout Rate: 0.250 Epoch 13: accuracy = 0.9955
Dropout Rate: 0.250 Epoch 14: accuracy = 0.9953
Dropout Rate: 0.250 Epoch 15: accuracy = 0.9945
Model 11 finished training
Dropout Rate: 0.275 Epoch 1: accuracy = 0.9861
Dropout Rate: 0.275 Epoch 2: accuracy = 0.9905
Dropout Rate: 0.275 Epoch 3: accuracy = 0.9912
Dropout Rate: 0.275 Epoch 4: accuracy = 0.9928
Dropout Rate: 0.275 Epoch 5: accuracy = 0.9934
Dropout Rate: 0.275 Epoch 6: accuracy = 0.9933
Dropout Rate: 0.275 Epoch 7: accuracy = 0.9926
Dropout Rate: 0.275 Epoch 8: accuracy = 0.9939
Dropout Rate: 0.275 Epoch 9: accuracy = 0.9940
Dropout Rate: 0.275 Epoch 10: accuracy = 0.9940
Dropout Rate: 0.275 Epoch 11: accuracy = 0.9947
Dropout Rate: 0.275 Epoch 12: accuracy = 0.9944
Dropout Rate: 0.275 Epoch 13: accuracy = 0.9951
Dropout Rate: 0.275 Epoch 14: accuracy = 0.9943
Dropout Rate: 0.275 Epoch 15: accuracy = 0.9949
Model 12 finished training
Dropout Rate: 0.300 Epoch 1: accuracy = 0.9873
Dropout Rate: 0.300 Epoch 2: accuracy = 0.9912
Dropout Rate: 0.300 Epoch 3: accuracy = 0.9901
Dropout Rate: 0.300 Epoch 4: accuracy = 0.9938
Dropout Rate: 0.300 Epoch 5: accuracy = 0.9917
Dropout Rate: 0.300 Epoch 6: accuracy = 0.9936
Dropout Rate: 0.300 Epoch 7: accuracy = 0.9936
Dropout Rate: 0.300 Epoch 8: accuracy = 0.9937
Dropout Rate: 0.300 Epoch 9: accuracy = 0.9944
Dropout Rate: 0.300 Epoch 10: accuracy = 0.9945
Dropout Rate: 0.300 Epoch 11: accuracy = 0.9946
Dropout Rate: 0.300 Epoch 12: accuracy = 0.9950
Dropout Rate: 0.300 Epoch 13: accuracy = 0.9948
Dropout Rate: 0.300 Epoch 14: accuracy = 0.9953
Dropout Rate: 0.300 Epoch 15: accuracy = 0.9949
Model 13 finished training
Dropout Rate: 0.325 Epoch 1: accuracy = 0.9858
Dropout Rate: 0.325 Epoch 2: accuracy = 0.9899
Dropout Rate: 0.325 Epoch 3: accuracy = 0.9917
Dropout Rate: 0.325 Epoch 4: accuracy = 0.9913
Dropout Rate: 0.325 Epoch 5: accuracy = 0.9924
Dropout Rate: 0.325 Epoch 6: accuracy = 0.9919
Dropout Rate: 0.325 Epoch 7: accuracy = 0.9914
Dropout Rate: 0.325 Epoch 8: accuracy = 0.9945
Dropout Rate: 0.325 Epoch 9: accuracy = 0.9943
Dropout Rate: 0.325 Epoch 10: accuracy = 0.9948
Dropout Rate: 0.325 Epoch 11: accuracy = 0.9933
Dropout Rate: 0.325 Epoch 12: accuracy = 0.9951
Dropout Rate: 0.325 Epoch 13: accuracy = 0.9956
Dropout Rate: 0.325 Epoch 14: accuracy = 0.9945
Dropout Rate: 0.325 Epoch 15: accuracy = 0.9937
Model 14 finished training
Dropout Rate: 0.350 Epoch 1: accuracy = 0.9846
Dropout Rate: 0.350 Epoch 2: accuracy = 0.9902
Dropout Rate: 0.350 Epoch 3: accuracy = 0.9883
Dropout Rate: 0.350 Epoch 4: accuracy = 0.9913
Dropout Rate: 0.350 Epoch 5: accuracy = 0.9933
Dropout Rate: 0.350 Epoch 6: accuracy = 0.9907
Dropout Rate: 0.350 Epoch 7: accuracy = 0.9923
Dropout Rate: 0.350 Epoch 8: accuracy = 0.9941
Dropout Rate: 0.350 Epoch 9: accuracy = 0.9945
Dropout Rate: 0.350 Epoch 10: accuracy = 0.9942
Dropout Rate: 0.350 Epoch 11: accuracy = 0.9945
Dropout Rate: 0.350 Epoch 12: accuracy = 0.9947
Dropout Rate: 0.350 Epoch 13: accuracy = 0.9949
Dropout Rate: 0.350 Epoch 14: accuracy = 0.9938
Dropout Rate: 0.350 Epoch 15: accuracy = 0.9938
Model 15 finished training
Dropout Rate: 0.375 Epoch 1: accuracy = 0.9828
Dropout Rate: 0.375 Epoch 2: accuracy = 0.9873
Dropout Rate: 0.375 Epoch 3: accuracy = 0.9905
Dropout Rate: 0.375 Epoch 4: accuracy = 0.9915
Dropout Rate: 0.375 Epoch 5: accuracy = 0.9920
Dropout Rate: 0.375 Epoch 6: accuracy = 0.9933
Dropout Rate: 0.375 Epoch 7: accuracy = 0.9935
Dropout Rate: 0.375 Epoch 8: accuracy = 0.9926
Dropout Rate: 0.375 Epoch 9: accuracy = 0.9936
Dropout Rate: 0.375 Epoch 10: accuracy = 0.9946
Dropout Rate: 0.375 Epoch 11: accuracy = 0.9945
Dropout Rate: 0.375 Epoch 12: accuracy = 0.9938
Dropout Rate: 0.375 Epoch 13: accuracy = 0.9941
Dropout Rate: 0.375 Epoch 14: accuracy = 0.9946
Dropout Rate: 0.375 Epoch 15: accuracy = 0.9956
Model 16 finished training
Dropout Rate: 0.400 Epoch 1: accuracy = 0.9877
Dropout Rate: 0.400 Epoch 2: accuracy = 0.9884
Dropout Rate: 0.400 Epoch 3: accuracy = 0.9904
Dropout Rate: 0.400 Epoch 4: accuracy = 0.9928
Dropout Rate: 0.400 Epoch 5: accuracy = 0.9916
Dropout Rate: 0.400 Epoch 6: accuracy = 0.9925
Dropout Rate: 0.400 Epoch 7: accuracy = 0.9933
Dropout Rate: 0.400 Epoch 8: accuracy = 0.9925
Dropout Rate: 0.400 Epoch 9: accuracy = 0.9939
Dropout Rate: 0.400 Epoch 10: accuracy = 0.9946
Dropout Rate: 0.400 Epoch 11: accuracy = 0.9950
Dropout Rate: 0.400 Epoch 12: accuracy = 0.9946
Dropout Rate: 0.400 Epoch 13: accuracy = 0.9958
Dropout Rate: 0.400 Epoch 14: accuracy = 0.9946
Dropout Rate: 0.400 Epoch 15: accuracy = 0.9950
Model 17 finished training
Dropout Rate: 0.425 Epoch 1: accuracy = 0.9857
Dropout Rate: 0.425 Epoch 2: accuracy = 0.9888
Dropout Rate: 0.425 Epoch 3: accuracy = 0.9901
Dropout Rate: 0.425 Epoch 4: accuracy = 0.9913
Dropout Rate: 0.425 Epoch 5: accuracy = 0.9916
Dropout Rate: 0.425 Epoch 6: accuracy = 0.9921
Dropout Rate: 0.425 Epoch 7: accuracy = 0.9925
Dropout Rate: 0.425 Epoch 8: accuracy = 0.9932
Dropout Rate: 0.425 Epoch 9: accuracy = 0.9940
Dropout Rate: 0.425 Epoch 10: accuracy = 0.9932
Dropout Rate: 0.425 Epoch 11: accuracy = 0.9941
Dropout Rate: 0.425 Epoch 12: accuracy = 0.9934
Dropout Rate: 0.425 Epoch 13: accuracy = 0.9941
Dropout Rate: 0.425 Epoch 14: accuracy = 0.9942
Dropout Rate: 0.425 Epoch 15: accuracy = 0.9942
Model 18 finished training
Dropout Rate: 0.450 Epoch 1: accuracy = 0.9795
Dropout Rate: 0.450 Epoch 2: accuracy = 0.9884
Dropout Rate: 0.450 Epoch 3: accuracy = 0.9918
Dropout Rate: 0.450 Epoch 4: accuracy = 0.9920
Dropout Rate: 0.450 Epoch 5: accuracy = 0.9922
Dropout Rate: 0.450 Epoch 6: accuracy = 0.9922
Dropout Rate: 0.450 Epoch 7: accuracy = 0.9938
Dropout Rate: 0.450 Epoch 8: accuracy = 0.9941
Dropout Rate: 0.450 Epoch 9: accuracy = 0.9948
Dropout Rate: 0.450 Epoch 10: accuracy = 0.9958
Dropout Rate: 0.450 Epoch 11: accuracy = 0.9941
Dropout Rate: 0.450 Epoch 12: accuracy = 0.9943
Dropout Rate: 0.450 Epoch 13: accuracy = 0.9951
Dropout Rate: 0.450 Epoch 14: accuracy = 0.9955
Dropout Rate: 0.450 Epoch 15: accuracy = 0.9952
Model 19 finished training
Dropout Rate: 0.475 Epoch 1: accuracy = 0.9850
Dropout Rate: 0.475 Epoch 2: accuracy = 0.9891
Dropout Rate: 0.475 Epoch 3: accuracy = 0.9906
Dropout Rate: 0.475 Epoch 4: accuracy = 0.9918
Dropout Rate: 0.475 Epoch 5: accuracy = 0.9912
Dropout Rate: 0.475 Epoch 6: accuracy = 0.9934
Dropout Rate: 0.475 Epoch 7: accuracy = 0.9943
Dropout Rate: 0.475 Epoch 8: accuracy = 0.9941
Dropout Rate: 0.475 Epoch 9: accuracy = 0.9946
Dropout Rate: 0.475 Epoch 10: accuracy = 0.9938
Dropout Rate: 0.475 Epoch 11: accuracy = 0.9953
Dropout Rate: 0.475 Epoch 12: accuracy = 0.9943
Dropout Rate: 0.475 Epoch 13: accuracy = 0.9950
Dropout Rate: 0.475 Epoch 14: accuracy = 0.9947
Dropout Rate: 0.475 Epoch 15: accuracy = 0.9962
Model 20 finished training'''